In [1]:
import os
import shutil
import random

# ==========================================
# 1. MAPEOS DE CLASES (TUS 5 CATEGORÍAS)
# 0: Organico | 1: Plastico | 2: Papel_Carton | 3: Metal | 4: Vidrio
# ==========================================

map_ds1 = {
    0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 
    10: 0, 11: 0, 12: 0, 13: 0, 28: 0, 29: 0, # Comida -> Organico
    14: 3, # Tin can -> Metal
    15: 1, # Bottle -> Plastico
    16: 2, 17: 2, 18: 2, # Papel/Carton
    21: 1  # Plastic bag -> Plastico
}

map_ds2 = {
    0: 0, # BIODEGRADABLE -> Organico
    1: 2, # CARDBOARD -> Papel_Carton
    2: 4, # GLASS -> Vidrio
    3: 3, # METAL -> Metal
    4: 2, # PAPER -> Papel_Carton
    5: 1  # PLASTIC -> Plastico
}

map_ds3 = {
    0: 2, # Paper
    1: 1, # Plastic
    2: 4, # Glass
    3: 3, # Metal
    4: 0  # Organic
}

# Basado en tus 4 clases: AluCan, Glass, PET, HDPEM
map_ds4 = {
    0: 3, # AluCan -> Metal
    1: 4, # Glass -> Vidrio
    2: 1, # PET -> Plastico
    3: 1  # HDPEM -> Plastico
}

# ==========================================
# 2. CREACIÓN DE DIRECTORIOS
# ==========================================
ruta_destino = r"C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\Dataset_Unificado"

# Limpiar si ya existía para evitar duplicados
if os.path.exists(ruta_destino):
    shutil.rmtree(ruta_destino)

for split in ['train', 'val']:
    os.makedirs(os.path.join(ruta_destino, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(ruta_destino, split, 'labels'), exist_ok=True)

# ==========================================
# 3. FUNCIONES DE PROCESAMIENTO
# ==========================================
def procesar_archivo_txt(path_origen, path_destino, diccionario):
    nuevas_lineas = []
    with open(path_origen, 'r') as f:
        for linea in f:
            partes = linea.strip().split()
            if not partes: continue
            clase_original = int(partes[0])
            if clase_original in diccionario:
                nueva_clase = diccionario[clase_original]
                nuevas_lineas.append(f"{nueva_clase} " + " ".join(partes[1:]) + "\n")
                
    if nuevas_lineas:
        with open(path_destino, 'w') as f_nuevo:
            f_nuevo.writelines(nuevas_lineas)
        return True
    return False

def procesar_dataset_normal(ruta_origen, diccionario, prefix):
    print(f"Procesando {prefix}...")
    for split in ['train', 'val']:
        ruta_img = os.path.join(ruta_origen, split, 'images')
        ruta_lbl = os.path.join(ruta_origen, split, 'labels')
        
        # Ajuste para carpetas llamadas 'valid'
        if not os.path.exists(ruta_img) and split == 'val':
            ruta_img = os.path.join(ruta_origen, 'valid', 'images')
            ruta_lbl = os.path.join(ruta_origen, 'valid', 'labels')
            
        if not os.path.exists(ruta_lbl): continue
            
        for archivo in os.listdir(ruta_lbl):
            if not archivo.endswith('.txt'): continue
            
            path_txt_orig = os.path.join(ruta_lbl, archivo)
            nuevo_nombre = f"{prefix}_{archivo}"
            path_txt_dest = os.path.join(ruta_destino, split, 'labels', nuevo_nombre)
            
            if procesar_archivo_txt(path_txt_orig, path_txt_dest, diccionario):
                nombre_base = archivo.replace('.txt', '')
                for ext in ['.jpg', '.png', '.jpeg', '.JPG']:
                    path_img_orig = os.path.join(ruta_img, nombre_base + ext)
                    if os.path.exists(path_img_orig):
                        shutil.copy(path_img_orig, os.path.join(ruta_destino, split, 'images', f"{prefix}_{nombre_base}{ext}"))
                        break

def procesar_dataset_mezclado(ruta_origen, diccionario, prefix):
    print(f"Procesando {prefix} y dividiendo 80/20...")
    archivos_txt = [f for f in os.listdir(ruta_origen) if f.endswith('.txt') and f != 'classes.txt' and f != 'labels.txt']
    random.shuffle(archivos_txt)
    
    punto_corte = int(len(archivos_txt) * 0.8)
    train_txts = archivos_txt[:punto_corte]
    val_txts = archivos_txt[punto_corte:]
    
    for lista, split in [(train_txts, 'train'), (val_txts, 'val')]:
        for archivo in lista:
            path_txt_orig = os.path.join(ruta_origen, archivo)
            nuevo_nombre = f"{prefix}_{archivo}"
            path_txt_dest = os.path.join(ruta_destino, split, 'labels', nuevo_nombre)
            
            if procesar_archivo_txt(path_txt_orig, path_txt_dest, diccionario):
                nombre_base = archivo.replace('.txt', '')
                for ext in ['.jpg', '.png', '.jpeg', '.JPG']:
                    path_img_orig = os.path.join(ruta_origen, nombre_base + ext)
                    if os.path.exists(path_img_orig):
                        shutil.copy(path_img_orig, os.path.join(ruta_destino, split, 'images', f"{prefix}_{nombre_base}{ext}"))
                        break

# ==========================================
# 4. BALANCEO FINAL DEL DATASET UNIFICADO
# ==========================================
def balancear_dataset():
    print("Balanceando exceso de Orgánico y Papel/Cartón a 10,000 imágenes máximo...")
    for split in ['train', 'val']:
        ruta_lbl = os.path.join(ruta_destino, split, 'labels')
        ruta_img = os.path.join(ruta_destino, split, 'images')
        
        candidatos_organico = []
        candidatos_papel = []
        
        for archivo in os.listdir(ruta_lbl):
            path_txt = os.path.join(ruta_lbl, archivo)
            clases = set()
            with open(path_txt, 'r') as f:
                for linea in f:
                    partes = linea.strip().split()
                    if partes: clases.add(int(partes[0]))
            
            if 1 in clases or 3 in clases or 4 in clases:
                continue
            if 0 in clases and len(clases) == 1:
                candidatos_organico.append(archivo)
            elif 2 in clases and len(clases) == 1:
                candidatos_papel.append(archivo)
                
        random.shuffle(candidatos_organico)
        random.shuffle(candidatos_papel)
        
        # Límite por carpeta (train suele ser más grande, dejamos 8000 en train y 2000 en val aprox)
        limite = 8000 if split == 'train' else 2000
        
        a_borrar = candidatos_organico[limite:] + candidatos_papel[limite:]
        
        for archivo in a_borrar:
            path_txt = os.path.join(ruta_lbl, archivo)
            os.remove(path_txt)
            nombre_base = archivo.replace('.txt', '')
            for ext in ['.jpg', '.png', '.jpeg', '.JPG']:
                path_img = os.path.join(ruta_img, nombre_base + ext)
                if os.path.exists(path_img):
                    os.remove(path_img)
                    break

# ==========================================
# 5. EJECUCIÓN
# ==========================================
ruta_ds1 = r"C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\dataset1"
ruta_ds2 = r"C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\dataset2\GARBAGE CLASSIFICATION"
ruta_ds3 = r"C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\dataset3"
ruta_ds4 = r"C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\dataset4\Images_of_Waste\YOLO_imgs"

procesar_dataset_normal(ruta_ds1, map_ds1, "ds1")
procesar_dataset_normal(ruta_ds2, map_ds2, "ds2")
procesar_dataset_normal(ruta_ds3, map_ds3, "ds3")
procesar_dataset_mezclado(ruta_ds4, map_ds4, "ds4")

balancear_dataset()

# Crear data.yaml final
yaml_content = """path: C:/Users/lomel/Documentos/HackaTec2026/ModelosBDWaste/Dataset_Unificado
train: train/images
val: val/images

nc: 5
names: ['Organico', 'Plastico', 'Papel_Carton', 'Metal', 'Vidrio']
"""
with open(os.path.join(ruta_destino, 'data.yaml'), 'w') as f:
    f.write(yaml_content)

print("¡Proceso maestro completado exitosamente! Tu dataset unificado, dividido y balanceado está listo.")

Procesando ds1...
Procesando ds2...
Procesando ds3...
Procesando ds4 y dividiendo 80/20...
Balanceando exceso de Orgánico y Papel/Cartón a 10,000 imágenes máximo...
¡Proceso maestro completado exitosamente! Tu dataset unificado, dividido y balanceado está listo.


In [1]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=r'C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\Dataset_Unificado\data.yaml',
    epochs=100, 
    imgsz=224, 
    device=0, 
    workers=8 
)

New https://pypi.org/project/ultralytics/8.4.159 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.46  Python-3.12.1 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\Dataset_Unificado\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=trai

In [1]:
import cv2
from ultralytics import YOLO

# 1. Carga tus mejores pesos (verifica que la ruta coincida con tu carpeta train5)
ruta_pesos = r'C:\Users\lomel\runs\detect\train-5\weights\best.pt'
# Nota: Si te marca error de que no encuentra el archivo, revisa si es 'train5' o 'train' a secas en tu explorador de archivos.
model = YOLO(ruta_pesos)

# 2. Encender la cámara web (el 0 es la cámara principal de tu laptop)
cap = cv2.VideoCapture(0)

print("¡Cámara encendida! Pon un poco de basura frente a la lente.")
print("Presiona la tecla 'q' en tu teclado para cerrar la ventana.")

while cap.isOpened():
    exito, frame = cap.read()
    if not exito:
        print("No se pudo acceder a la cámara.")
        break

    # 3. Hacer la predicción
    # Le ponemos conf=0.65 para que solo te muestre cosas de las que está súper seguro
    resultados = model(frame, conf=0.65, verbose=False)

    # (Opcional) Imprimir en consola lo que está viendo para tu futuro ESP32
    for resultado in resultados:
        for caja in resultado.boxes:
            nombre = model.names[int(caja.cls[0])]
            confianza = float(caja.conf[0]) * 100
            print(f"Detectado: {nombre} al {confianza:.1f}%")

    # 4. YOLO dibuja mágicamente los recuadros y los nombres sobre el video
    frame_anotado = resultados[0].plot()

    # 5. Mostrar la ventana de video
    cv2.imshow("Visión del Bote Inteligente", frame_anotado)

    # 6. Escuchar si presionas la tecla 'q' para salir
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Apagar la cámara y cerrar la ventana limpiamente
cap.release()
cv2.destroyAllWindows()

¡Cámara encendida! Pon un poco de basura frente a la lente.
Presiona la tecla 'q' en tu teclado para cerrar la ventana.
Detectado: Plastico al 73.1%
Detectado: Organico al 71.0%
Detectado: Organico al 67.6%
Detectado: Organico al 65.9%
Detectado: Organico al 69.3%
Detectado: Papel_Carton al 96.4%
Detectado: Papel_Carton al 96.3%
Detectado: Papel_Carton al 87.4%
Detectado: Papel_Carton al 88.4%
Detectado: Papel_Carton al 94.9%
Detectado: Papel_Carton al 97.5%
Detectado: Papel_Carton al 95.4%
Detectado: Papel_Carton al 95.4%
Detectado: Papel_Carton al 95.7%
Detectado: Papel_Carton al 95.0%
Detectado: Papel_Carton al 96.5%
Detectado: Papel_Carton al 95.9%
Detectado: Papel_Carton al 95.8%
Detectado: Papel_Carton al 94.6%
Detectado: Papel_Carton al 86.8%
Detectado: Papel_Carton al 91.2%
Detectado: Papel_Carton al 91.2%
Detectado: Papel_Carton al 75.7%
Detectado: Papel_Carton al 69.5%
Detectado: Papel_Carton al 92.8%
Detectado: Papel_Carton al 96.5%
Detectado: Papel_Carton al 96.6%
Detectado